# Group Assignment - Part A: Sentence Probability using Bigram Language Models (Q4)

This notebook implements Unsmoothed Bigram (MLE) and Smoothed Bigram (Laplace) language models using the three training sentences provided in Data_3.txt. The target sentence is kept separate from the training corpus. The implementation follows the workflow introduced in Lab 03 (Tokenization) and Lab 07 (Language Modelling).

## Environment Setup

### Environment Dependencies

First, import the required libraries for tokenization, bigram generation, language model construction and probability computation.

In [1]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.util import bigrams, everygrams
from nltk.lm.preprocessing import padded_everygram_pipeline, pad_both_ends
from nltk.lm import MLE, Laplace
import pandas as pd

nltk.download("punkt")

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/jeffrey1108/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

## Step 4.1: Load the Training Corpus

Only the three sentences listed under **Training Corpus** are loaded for model training. The sentence shown after the calculation instruction is the target sentence, so it must not be included in the training corpus.

In [2]:
with open("Data_3.txt", "r", encoding="utf-8") as file:
    raw_text = file.read()

print(raw_text)

# Keep only the text before the target-sentence instruction.
training_text = raw_text.split("Calculate sentence probability", 1)[0]

corpus = [
    line.strip()
    for line in training_text.splitlines()
    if line.strip().startswith("<s>") and line.strip().endswith("</s>")
]

print("\nTraining Corpus")
for sentence in corpus:
    print(sentence)

Training Corpus
~~~~~~~~~~~~~
<s> He read a book </s>
<s> I read a different book </s>
<s> He read a book by Danielle </s>

Calculate sentence probability for the following sentence
~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
<s> I read a book by Danielle </s>

Training Corpus
<s> He read a book </s>
<s> I read a different book </s>
<s> He read a book by Danielle </s>


## Step 4.2: Text Pre-processing

Each training sentence is tokenized while preserving its boundary markers for display. A second list, `model_ready_tokens`, removes the existing `<s>` and `</s>` markers before model training because NLTK adds one correct set of padding markers automatically. This prevents duplicated sentence boundaries.

In [3]:
tokenized_text = [sentence.split() for sentence in corpus]

model_ready_tokens = [
    tokens[1:-1]
    if tokens[0] == "<s>" and tokens[-1] == "</s>"
    else tokens
    for tokens in tokenized_text
]

print("Tokenized Sentences\n")
for i, sentence in enumerate(tokenized_text, start=1):
    print(f"Sentence {i}:")
    print(sentence)
    print()

Tokenized Sentences

Sentence 1:
['<s>', 'He', 'read', 'a', 'book', '</s>']

Sentence 2:
['<s>', 'I', 'read', 'a', 'different', 'book', '</s>']

Sentence 3:
['<s>', 'He', 'read', 'a', 'book', 'by', 'Danielle', '</s>']



# Step 4.3: Generate Bigrams

Each displayed training sentence is converted into consecutive word pairs. The target sentence is not present, so these bigrams represent only the three intended training sentences.

In [4]:
print("Generated Bigrams\n")

for i, sentence in enumerate(tokenized_text, start=1):

    print(f"Sentence {i}")

    bg = list(bigrams(sentence))

    print(bg)
    print()

Generated Bigrams

Sentence 1
[('<s>', 'He'), ('He', 'read'), ('read', 'a'), ('a', 'book'), ('book', '</s>')]

Sentence 2
[('<s>', 'I'), ('I', 'read'), ('read', 'a'), ('a', 'different'), ('different', 'book'), ('book', '</s>')]

Sentence 3
[('<s>', 'He'), ('He', 'read'), ('read', 'a'), ('a', 'book'), ('book', 'by'), ('by', 'Danielle'), ('Danielle', '</s>')]



# Step 4.4: Generate Everygrams

Everygrams are generated from `model_ready_tokens`. NLTK adds one `<s>` and one `</s>` marker to each sentence, allowing the model to learn the required unigram and bigram information without duplicating the boundary markers.

In [5]:
print("Everygrams\n")

for sentence in model_ready_tokens:
    padded = list(pad_both_ends(sentence, n=2))
    print(list(everygrams(padded, max_len=2)))
    print()

Everygrams

[('<s>',), ('<s>', 'He'), ('He',), ('He', 'read'), ('read',), ('read', 'a'), ('a',), ('a', 'book'), ('book',), ('book', '</s>'), ('</s>',)]

[('<s>',), ('<s>', 'I'), ('I',), ('I', 'read'), ('read',), ('read', 'a'), ('a',), ('a', 'different'), ('different',), ('different', 'book'), ('book',), ('book', '</s>'), ('</s>',)]

[('<s>',), ('<s>', 'He'), ('He',), ('He', 'read'), ('read',), ('read', 'a'), ('a',), ('a', 'book'), ('book',), ('book', 'by'), ('by',), ('by', 'Danielle'), ('Danielle',), ('Danielle', '</s>'), ('</s>',)]



# Step 4.5: Train the Unsmoothed Bigram Language Model (MLE)

The **Maximum Likelihood Estimation (MLE)** model is trained using the three model-ready sentences. It estimates each conditional probability directly from the observed bigram frequency.

In [6]:
n = 2

train_data, vocab_data = padded_everygram_pipeline(n, model_ready_tokens)

mle_model = MLE(n)
mle_model.fit(train_data, vocab_data)

print("Unsmoothed Bigram Language Model trained successfully.")

Unsmoothed Bigram Language Model trained successfully.


# Step 4.6: Train the Smoothed Bigram Language Model (Laplace)

The **Laplace (Add-One) Language Model** is trained using the same three model-ready sentences. NLTK uses a vocabulary size of 11, consisting of the 10 observed corpus tokens plus the `<UNK>` token used for unknown words.

In [7]:
train_data, vocab_data = padded_everygram_pipeline(n, model_ready_tokens)

laplace_model = Laplace(n)
laplace_model.fit(train_data, vocab_data)

print("Laplace Bigram Language Model trained successfully.")

Laplace Bigram Language Model trained successfully.


# Step 4.7: Prepare the Target Sentence

The target sentence is tokenized, padded with sentence boundary markers, and converted into bigrams before calculating its sentence probability.

In [8]:
target_sentence = "I read a book by Danielle"

target_tokens = word_tokenize(target_sentence)

print("Target Tokens")

print(target_tokens)

padded_target = list(pad_both_ends(target_tokens, n=2))

print("\nPadded Sentence")

print(padded_target)

target_bigrams = list(bigrams(padded_target))

print("\nTarget Sentence Bigrams")

print(target_bigrams)

Target Tokens
['I', 'read', 'a', 'book', 'by', 'Danielle']

Padded Sentence
['<s>', 'I', 'read', 'a', 'book', 'by', 'Danielle', '</s>']

Target Sentence Bigrams
[('<s>', 'I'), ('I', 'read'), ('read', 'a'), ('a', 'book'), ('book', 'by'), ('by', 'Danielle'), ('Danielle', '</s>')]


# Step 4.8: Compute Sentence Probability

The target sentence probability is calculated by multiplying the conditional probability of its seven bigrams. Both models use the same target bigrams and the same three-sentence training corpus as the manual calculation.

In [9]:
mle_probability = 1.0
laplace_probability = 1.0

results = []

for bg in target_bigrams:

    history = (bg[0],)

    word = bg[1]

    mle_score = mle_model.score(word, history)

    laplace_score = laplace_model.score(word, history)

    mle_probability *= mle_score

    laplace_probability *= laplace_score

    results.append([
        history[0],
        word,
        mle_score,
        laplace_score
    ])

results_df = pd.DataFrame(
    results,
    columns=[
        "History",
        "Word",
        "MLE Probability",
        "Laplace Probability"
    ]
)

results_df

,History,Word,MLE Probability,Laplace Probability
0,<s>,I,0.333333,0.142857
1,I,read,1.000000,0.166667
2,read,a,1.000000,0.285714
3,a,book,0.666667,0.214286
4,book,by,0.333333,0.142857
5,by,Danielle,1.000000,0.166667
6,Danielle,</s>,1.000000,0.166667


# Step 4.9: Display Sentence Probabilities

The overall probability is obtained by multiplying all seven conditional probabilities. The displayed results can now be compared directly with the corrected manual calculations.

In [10]:
print("Vocabulary Size (including <UNK>):", len(laplace_model.vocab))

print("\nSentence:")
print("<s> I read a book by Danielle </s>")

print("\nUnsmoothed (MLE) Sentence Probability")
print(mle_probability)

print("\nSmoothed (Laplace) Sentence Probability")
print(laplace_probability)

Vocabulary Size (including <UNK>): 11

Sentence:
<s> I read a book by Danielle </s>

Unsmoothed (MLE) Sentence Probability
0.07407407407407407

Smoothed (Laplace) Sentence Probability
5.784626775880419e-06


# Step 4.10: Compare the Results

| Model | Manual probability | Python probability | Match |
|---|---:|---:|:---:|
| Unsmoothed Bigram (MLE) | 0.0740740741 | 0.0740740741 | Yes |
| Smoothed Bigram (Laplace) | 5.7846268 × 10⁻⁶ | 5.7846268 × 10⁻⁶ | Yes |

# Conclusion

The corrected implementation trains both models on only the three sentences listed under **Training Corpus**. The target sentence is excluded from training, and the existing boundary markers are removed before NLTK adds one set of padding markers. As a result, the Python calculations now match the manual calculations: the MLE probability is **0.0740740741**, while the Laplace probability is approximately **5.7846268 × 10⁻⁶** using NLTK's vocabulary size of 11. Laplace produces the smaller value because add-one smoothing distributes probability mass across all vocabulary items while still assigning a non-zero probability to unseen bigrams.